<a href="https://colab.research.google.com/github/cbishop4/MSE7530/blob/main/ClassNotebooks/F2026/Lecture11_InClass_20261007.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# In-Class Hands-on AFM analysis

In [ ]:
!pip install uv -q
!uv pip install pySPM --system -q


In [ ]:
import pySPM as spm
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Download the AFM data from the public repo
!git clone https://github.com/cbishop4/Alahe2026.git

# Task 1. Load metadata.
The data shared in the public repo are raw Bruker AFM files. We'll examine what these files contain. Our objectives are to find:  
1. The images contained in the different AFM channels.  
2. The metadata the AFM recorded.  
3. A method (function) to pull the relevant metadata parameters.

### Work with a single file.  
First we will load a single file and see what it contains.

In [ ]:
rawAFM = '/content/Alahe2026/AFMdata/0p08nms/MAH26_01.030'
scan = spm.Bruker(rawAFM)
scan

In [ ]:
scan.list_channels()

### Explore the channels.

In [ ]:
ht_chan = scan.get_channel('Height')
ht_px_vals = ht_chan.pixels
print(f'The retrieved pixels is a {type(ht_px_vals)} of shape {ht_px_vals.shape}')

We can use `plt.imshow()` to look at the 2D array. We can put a colorbar on it too.

In [ ]:
plt.imshow(ht_px_vals)
plt.colorbar(label='Height (nm)')

An alternative formulation, which will be helpful when we start plotting in batch, is below.

In [ ]:
fig, ax = plt.subplots()
cb = ax.imshow(ht_px_vals)
fig.colorbar(cb, ax=ax,label='Height (nm)')

<font color='blue'>Do the same for the "Phase" and "Amplitude Error" channels.</font>

In [ ]:
fig, ax = plt.subplots(1,3)
cb1 = ax[0].imshow(ht_px_vals)
fig.colorbar(cb1, ax=ax[0],label='Height (nm)',location='top')
cb2 = ax[1].imshow(___)
fig.colorbar(cb2, ax=ax[1],label='Phase (degrees)',location='top')
cb3 = ax[2].imshow(___)
fig.colorbar(cb3, ax=ax[2],label='Amplitude Error (nm)',location='top')

### Look at the metadata.  
The cell below shows the metadata for each channel. Many of these entries are acquisition parameters recorded inside the instrument.

In [ ]:
scan.layers

We want to know how large the image is and how many nm each pixel corresponds to. We therefore need to pick out the image **resolution** and **scan size**.

In [ ]:
scan_size = scan.layers[0][b'Scan Size']
num_lines = scan.layers[0][b'Number of lines']
print(f'Our raw strings from the metadata are {scan_size} and {num_lines}')

We'll use the `re` package (built into Python), which contains "regular expression" operations.

In [ ]:
import re

In [ ]:
scan_size = scan.layers[0][b'Scan Size']
num_lines = scan.layers[0][b'Number of lines']
ss = float(re.findall(r'\d+(?:\.\d+)?', str(scan_size[0]))[0])
nlines = int(re.findall(r'\d+', str(num_lines[0]))[0])
print(f'We used re to find the scan size {ss} as a {type(ss)} and the number of lines {nlines} as a {type(nlines)}')

One thing we know about our instrument: if the scan size pulled out is between 1 and 50, it is reported in $\mu m$. If it is larger, such as 500, it is reported in $nm$. We can handle that with the following block:

In [ ]:
if ss < 45.:
  ss_nm = ss*1000.
else:
  ss_nm = ss

We can find the nm per pixel by dividing the scan size by the resolution. <font color='blue'>Do this below to make a variable called `px_nm`.</font>

### <font color='blue'>Write a function to return the scan size in nm, the resolution, and the size of a pixel in nm.</font>  
Take 5–10 minutes to do this; then we will regroup as a class and discuss how to do it on the board. Hints:
1. The input to your function can start all the way back at the filename, or at the `scan`. Make sure you start with something that still contains the metadata.  
2. All the code you need can be copy-pasted from above.  
3. I suggest returning 3 separate variables: `ss_nm`, `nlines`, and `px_nm`. This is for uniformity across the class.  
4. Name your function `pull_params(filename, print_explicit=False)`, where `print_explicit=True` prints the three values. The bonus section below calls it this way.

# Task 2. Roughness quantification.

Together on the board, we will work through the mathematical definitions of the five measures of roughness that are schematically illustrated in the slides. <font color='blue'>Then, below, write a cell that calculates each of these for a 2D NumPy array, using the variable names `R_q`, `R_a`, `R_p`, `R_v`, and `R_T` so that the print cell below works.</font>

In [ ]:
print(f'Rq: {R_q} nm')
print(f'Ra: {R_a} nm')
print(f'Rp: {R_p} nm')
print(f'Rv: {R_v} nm')
print(f'RT: {R_T} nm')

<font color='blue'>Now turn these into user-defined functions. You can write a separate function for each, or a single function that returns all five (i.e., `return R_q, R_a, R_p, R_v, R_T`). The loops later in the notebook use a single function called `calc_Rmeasures(heights)`, so use that name for the combined version.</font>

# Task 3. Compare across processing conditions (deposition rates).  
Now we will do something analogous to what you'll be doing in the project: comparing some parameters to the processing condition. I'll start by listing a few things that will help you move through the data automatically. We'll go through these together and take notes on what exactly is happening.

In [ ]:
# deposition rate labels (used in folder names) and the same rates as numbers, in nm/s
rate_codes = ['0p8','0p4','0p08','0p04','0p008']
rs_floats = np.asarray([0.8, 0.4, 0.08, 0.04, 0.008])

# one representative raw AFM file for each deposition rate
filelist = ['/content/Alahe2026/AFMdata/0p8nms/MAH27.037',
            '/content/Alahe2026/AFMdata/0p4nms/MAH23_01.012',
            '/content/Alahe2026/AFMdata/0p08nms/MAH26_01.030',
            '/content/Alahe2026/AFMdata/0p04nms/MAH25_01.024',
            '/content/Alahe2026/AFMdata/0p008nms/MAH29.047']

First, let's just plot all of these.

In [ ]:
channel_name = 'Height'
fig, ax = plt.subplots(1,5,figsize=(20,3))
for i,f in enumerate(filelist):
  scan = spm.Bruker(f)
  channel1 = scan.get_channel(channel_name)
  channel_corr = channel1.correct_lines(inline=False)
  channel = channel_corr.pixels
  ax[i].imshow(channel)
  ax[i].set_title(f'R_dep = {rs_floats[i]} nm/s')

These are all 2000 x 2000 nm images. For each file, calculate the five roughness measures and store them in the lists below; then run the plotting cell that follows.

In [ ]:
R_qs = []; R_as = []; R_ps = []; R_vs = []; R_Ts = []
for f in filelist:
  pass  # your code here

In [ ]:
fig, ax = plt.subplots(1,5,figsize=(15,3))
plt.subplots_adjust(wspace=0.3)
ax[0].plot(rs_floats, R_qs,'-o')
ax[1].plot(rs_floats, R_as,'-o')
ax[2].plot(rs_floats, R_ps,'-o')
ax[3].plot(rs_floats, R_vs,'-o')
ax[4].plot(rs_floats, R_Ts,'-o')

for a in ax:
  a.semilogx()
  a.set_xlabel('$R_{dep}$ (nm/s)')
ax[0].set_ylabel('Rq (nm)')
ax[1].set_ylabel('Ra (nm)')
ax[2].set_ylabel('Rp (nm)')
ax[3].set_ylabel('Rv (nm)')
ax[4].set_ylabel('RT (nm)')

# Task 4. Analysis in reciprocal space.

In [ ]:
fname = '/content/Alahe2026/AFMdata/0p8nms/MAH27.037'
scan = spm.Bruker(fname)
channel1 = scan.get_channel('Height')
channel_corr = channel1.correct_lines(inline=False)
channel = channel_corr.pixels

In [ ]:
from scipy.fft import fft2

In [ ]:
# Perform 2D FFT on the 'channel' data
F = fft2(channel)

# Shift the zero-frequency component to the center
F_shifted = np.fft.fftshift(F)

# Calculate the power spectrum (magnitude squared of the FFT result)
power_spectrum = np.abs(F_shifted)**2

# Get image dimensions (assuming a square image)
N = channel.shape[0]

# Generate radial frequency coordinates
# Frequencies are in cycles/unit_distance. Here, unit_distance is nm.
# d=px_nm converts pixel indices to real-world spatial frequencies in 1/nm.
freq_x = np.fft.fftfreq(N, d=px_nm)
freq_y = np.fft.fftfreq(N, d=px_nm)

# Shift frequencies to match the shifted FFT
freq_x_shifted = np.fft.fftshift(freq_x)
freq_y_shifted = np.fft.fftshift(freq_y)

# Create a 2D meshgrid for radial spatial frequencies
kx, ky = np.meshgrid(freq_x_shifted, freq_y_shifted)
q = np.sqrt(kx**2 + ky**2) # Radial spatial frequency

# Flatten radial frequencies and power spectrum for binning
q_flat = q.flatten()
ps_flat = power_spectrum.flatten()

# Sort by radial frequency to facilitate binning
sorted_indices = np.argsort(q_flat)
q_sorted = q_flat[sorted_indices]
ps_sorted = ps_flat[sorted_indices]

# Define radial bins for 1D PSD
# Exclude q=0 for log-spaced bins, as log(0) is undefined
min_q = q_sorted[q_sorted > 0].min()
max_q = q_sorted.max()
num_bins = 100 # Number of bins for the 1D PSD

# Create log-spaced bins
bins = np.logspace(np.log10(min_q), np.log10(max_q), num_bins + 1)

# Calculate the mean power spectral density within each bin
bin_centers = (bins[:-1] + bins[1:]) / 2
psd_1d = np.zeros(num_bins)

for i in range(num_bins):
    mask = (q_sorted >= bins[i]) & (q_sorted < bins[i+1])
    if np.any(mask):
        psd_1d[i] = np.mean(ps_sorted[mask])
    else:
        psd_1d[i] = np.nan # Mark empty bins as NaN

# Filter out NaN values from empty bins if any
valid_indices = ~np.isnan(psd_1d)
q_1d_psd = bin_centers[valid_indices]
psd_1d = psd_1d[valid_indices]

print(f'Shape of 1D spatial frequencies (q): {q_1d_psd.shape}')
print(f'Shape of 1D Power Spectral Density: {psd_1d.shape}')

# Plot the 1D Power Spectral Density
plt.figure(figsize=(8, 6))
plt.loglog(q_1d_psd, psd_1d, 'o-', markersize=4)
plt.xlabel('Spatial Frequency q (1/nm)')
plt.ylabel('Power Spectral Density (nm$^2$)')
plt.title('1D Power Spectral Density from AFM Image')
plt.grid(True, which="both", ls="-", alpha=0.6)
plt.tight_layout()
plt.show()

<font color='blue'>Repeat this for each deposition rate. What do you see?</font>

# Bonus: Using `glob` to automatically go through every image in each folder.  
`glob` matches pathnames to automatically find different files.

Load all the files and take a look.

In [ ]:
import glob

In [ ]:
for f in glob.glob('/content/Alahe2026/AFMdata/0p08nms/M*'):
  print(f)

In [ ]:
channel_name = 'Height'
fig, ax = plt.subplots(1,6,figsize=(20,3))
for i,f in enumerate(glob.glob('/content/Alahe2026/AFMdata/0p08nms/M*')):
  scan = spm.Bruker(f)
  channel1 = scan.get_channel(channel_name)
  channel_corr = channel1.correct_lines(inline=False)
  channel = channel_corr.pixels
  ax[i].imshow(channel)

In [ ]:
# pull metadata
scan = spm.Bruker('/content/Alahe2026/AFMdata/0p08nms/MAH26_01.030')

In [ ]:
pp = pull_params('/content/Alahe2026/AFMdata/0p08nms/MAH26_01.030',print_explicit=True)

In [ ]:
pp

In [ ]:
rate_codes = ['0p8','0p4','0p08','0p04','0p008']
rs_floats = np.asarray([0.8, 0.4, 0.08, 0.04, 0.008])
Rq_asr = np.zeros(5); Ra_asr = np.zeros(5); Rp_asr = np.zeros(5); Rv_asr = np.zeros(5); RT_asr = np.zeros(5)
i = 0
for r in rate_codes:
  scan_sizes = []; R_qs = []; R_as = []; R_ps = []; R_vs = []; R_Ts = []
  for f in glob.glob(f'/content/Alahe2026/AFMdata/{r}nms/M*'):
    scan = spm.Bruker(f)
    channel1 = scan.get_channel(channel_name)
    channel_corr = channel1.correct_lines(inline=False)
    channel = channel_corr.pixels
    ss_nm, nlines, px_nm = pull_params(f,print_explicit=False)
    R_q, R_a, R_p, R_v, R_T = calc_Rmeasures(channel)
    scan_sizes.append(ss_nm)
    R_qs.append(R_q)
    R_as.append(R_a)
    R_ps.append(R_p)
    R_vs.append(R_v)
    R_Ts.append(R_T)
  R_qs = np.asarray(R_qs)
  R_as = np.asarray(R_as)
  R_ps = np.asarray(R_ps)
  R_vs = np.asarray(R_vs)
  R_Ts = np.asarray(R_Ts)
  Rq_asr[i] = np.mean(R_qs)
  Ra_asr[i] = np.mean(R_as)
  Rp_asr[i] = np.mean(R_ps)
  Rv_asr[i] = np.mean(R_vs)
  RT_asr[i] = np.mean(R_Ts)
  i += 1

In [ ]:
plt.plot(rs_floats, Rq_asr,'o')
plt.semilogx()

# Bonus: scan size correlation

See if there's any correlation to scan size

In [ ]:
channel_name = 'Height'
fig, ax = plt.subplots(1,6,figsize=(20,3))
scan_sizes = []; R_qs = []; R_as = []; R_ps = []; R_vs = []; R_Ts = []
for i,f in enumerate(glob.glob('/content/Alahe2026/AFMdata/0p08nms/M*')):
  scan = spm.Bruker(f)
  channel1 = scan.get_channel(channel_name)
  channel_corr = channel1.correct_lines(inline=False)
  channel = channel_corr.pixels
  ax[i].imshow(channel)
  ss_nm, nlines, px_nm = pull_params(f,print_explicit=False)
  R_q, R_a, R_p, R_v, R_T = calc_Rmeasures(channel)
  scan_sizes.append(ss_nm)
  R_qs.append(R_q)
  R_as.append(R_a)
  R_ps.append(R_p)
  R_vs.append(R_v)
  R_Ts.append(R_T)
  ax[i].set_title(f'Scan size: {ss_nm} nm')

In [ ]:
plt.plot(R_qs,'o',label='Rq')
plt.plot(R_as,'s',label='Ra')
plt.plot(R_ps,'>',label='Rp')
plt.plot(R_vs,'*',label='Rv')
plt.plot(R_Ts,'^',label='RT')
plt.legend()

In [ ]:
plt.plot(scan_sizes, R_qs,'o',label='Rq')
plt.plot(scan_sizes, R_as,'s',label='Ra')
plt.plot(scan_sizes, R_ps,'>',label='Rp')
plt.plot(scan_sizes, R_vs,'*',label='Rv')
plt.plot(scan_sizes, R_Ts,'^',label='RT')
plt.legend()

In [ ]:
plt.plot(scan_sizes, R_qs,'o',label='Rq')
plt.plot(scan_sizes, R_as,'s',label='Ra')

### Testing for another rate

In [ ]:
channel_name = 'Height'
fig, ax = plt.subplots(1,7,figsize=(20,3))
scan_sizes = []; R_qs = []; R_as = []; R_ps = []; R_vs = []; R_Ts = []
for i,f in enumerate(glob.glob('/content/Alahe2026/AFMdata/0p04nms/M*')):
  scan = spm.Bruker(f)
  channel1 = scan.get_channel(channel_name)
  channel_corr = channel1.correct_lines(inline=False)
  channel = channel_corr.pixels
  ax[i].imshow(channel)
  ss_nm, nlines, px_nm = pull_params(f,print_explicit=False)
  R_q, R_a, R_p, R_v, R_T = calc_Rmeasures(channel)
  scan_sizes.append(ss_nm)
  R_qs.append(R_q)
  R_as.append(R_a)
  R_ps.append(R_p)
  R_vs.append(R_v)
  R_Ts.append(R_T)
  ax[i].set_title(f'Scan size: {ss_nm} nm')

In [ ]:
plt.plot(R_qs,'o',label='Rq')
plt.plot(R_as,'s',label='Ra')
plt.plot(R_ps,'>',label='Rp')
plt.plot(R_vs,'*',label='Rv')
plt.plot(R_Ts,'^',label='RT')
plt.legend()

In [ ]:
plt.plot(scan_sizes, R_qs,'o',label='Rq')
plt.plot(scan_sizes, R_as,'s',label='Ra')
plt.plot(scan_sizes, R_ps,'>',label='Rp')
plt.plot(scan_sizes, R_vs,'*',label='Rv')
plt.plot(scan_sizes, R_Ts,'^',label='RT')
plt.legend()